# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

Dataset: _Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution_.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
Below, we inspect what record sets, fields, and columns exist in the dataset, referencing each by its `@id`.

In [ ]:
# List all available record sets
print("Available record sets and fields by @id:")
record_sets = []
for record_set in dataset.record_sets:
    print(f'- Record set @id: {record_set.id}  name: {record_set.name}')
    record_sets.append(record_set.id)
    for field in record_set.fields:
        print(f'    - Field @id: {field.id}  name: {field.name}  dataType: {getattr(field, "data_type", "N/A")}')
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f'        - Column @id: {col.id}  name: {col.name}')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id` field.

In [ ]:
# Extract data from each record set into a DataFrame, using their @id
dfs = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f'Loaded data for record set: {rs_id}')
    print(f'Columns: {df.columns.tolist()}')
    print(f'Number of records: {len(df)}\n')

# View the first record set's data
if record_sets:
    selected_rs = record_sets[0]
    display(dfs[selected_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply simple data processing: filtering records based on a numeric field, normalization, and grouping by a key attribute. All fields and columns are referenced by their `@id`.

In [ ]:
# Identify a numeric field by @id (see overview above and adjust as appropriate)
# For illustration, let's assume the record set contains a numeric field with @id 'https://api.app.sen.science/frontiers/7862866/field-diagnosis_interval_months'
# and a group field with @id 'https://api.app.sen.science/frontiers/7862866/field-msi_status'
# Replace below with the correct @id values derived from your data overview outputs.
record_set_id = selected_rs
# Example field @ids (replace with the actual IDs as shown in section 2)
numeric_field_id = None
group_field_id = None
for record_set in dataset.record_sets:
    if record_set.id == record_set_id:
        # Try to find a numeric field
        for field in record_set.fields:
            if hasattr(field, 'data_type') and str(field.data_type).lower() in ('integer','float','number'):
                numeric_field_id = field.id
            if not group_field_id and field.name.lower().startswith('msi'):
                group_field_id = field.id
        break

print(f"Numeric field selected for EDA (by @id): {numeric_field_id}")
print(f"Grouping field (by @id): {group_field_id}")

df = dfs[record_set_id]
# If the fields have @id as the columns, but the loaded dataframe's columns are the field 'name',
# try both. Use the field's name as key if @id not found in dataframe columns.

field_id_to_name = {f.id: f.name for r in dataset.record_sets for f in r.fields}
num_col = None
grp_col = None
for key in (numeric_field_id, field_id_to_name.get(numeric_field_id)):
    if key in df.columns:
        num_col = key
        break
for key in (group_field_id, field_id_to_name.get(group_field_id)):
    if key in df.columns:
        grp_col = key
        break

if num_col is not None:
    # Try filtering on the numeric field
    print(f"\nFiltering records with {num_col} > 10:")
    try:
        filtered = df[df[num_col].astype(float) > 10].copy()
    except Exception as e:
        print('Conversion to float failed, skipping filtering.')
        filtered = df.copy()
    print(filtered.head())
    # Normalize
    filtered[f"{num_col}_normalized"] = (filtered[num_col].astype(float) - filtered[num_col].astype(float).mean()) / filtered[num_col].astype(float).std()
    print(f"\nNormalized {num_col} for filtered records:")
    print(filtered[[num_col, f"{num_col}_normalized"]].head())
    # Grouping
    if grp_col is not None:
        grouped = filtered.groupby(grp_col)[num_col].mean()
        print(f"\nGrouped data by {grp_col} (means):")
        print(grouped.head())
else:
    print('No numeric field identified for analysis.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Below is a sample histogram for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if num_col is not None and num_col in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[num_col].dropna().astype(float), kde=True)
    plt.title(f"Distribution of {num_col}")
    plt.xlabel(num_col)
    plt.ylabel('Count')
    plt.show()

    if grp_col is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[grp_col], y=df[num_col].astype(float))
        plt.title(f"{num_col} grouped by {grp_col}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Cannot visualize: numeric column not available.')

## 6. Conclusion
This notebook demonstrated the loading, exploration, and initial processing of the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library. We listed dataset entities by `@id`, loaded data to pandas DataFrames, and performed sample EDA such as filtering, normalization, grouping, and visualization. For more detailed analysis or reporting, select and reference fields and record sets by their `@id` as demonstrated here. 